In [3]:

def transpose(matrix):
    """Transpose sebuah matriks 2D."""
    rows = len(matrix)
    cols = len(matrix[0])
    return [[matrix[r][c] for r in range(rows)] for c in range(cols)]


def mat_mul(A, B):
    """Perkalian dua matriks."""
    rows_A, cols_A = len(A), len(A[0])
    rows_B, cols_B = len(B), len(B[0])
    assert cols_A == rows_B, "Dimensi matriks tidak cocok untuk perkalian."
    C = [[0.0] * cols_B for _ in range(rows_A)]
    for i in range(rows_A):
        for j in range(cols_B):
            for k in range(cols_A):
                C[i][j] += A[i][k] * B[k][j]
    return C


def vec_dot(a, b):
    """Dot product dua vektor."""
    return sum(x * y for x, y in zip(a, b))


def vec_norm(v):
    """Norm (panjang) sebuah vektor."""
    return sum(x ** 2 for x in v) ** 0.5


def vec_normalize(v):
    """Normalisasi vektor menjadi unit vector."""
    norm = vec_norm(v)
    if norm == 0:
        return v
    return [x / norm for x in v]


def vec_scale(v, scalar):
    """Perkalian vektor dengan skalar."""
    return [x * scalar for x in v]


def vec_sub(a, b):
    """Pengurangan dua vektor."""
    return [x - y for x, y in zip(a, b)]


def mat_vec_mul(A, v):
    """Perkalian matriks dengan vektor."""
    return [vec_dot(row, v) for row in A]


def outer_product(a, b):
    """Outer product dua vektor → menghasilkan matriks."""
    return [[ai * bj for bj in b] for ai in a]


def mat_sub(A, B):
    """Pengurangan dua matriks elemen per elemen."""
    return [[A[i][j] - B[i][j] for j in range(len(A[0]))] for i in range(len(A))]


def mat_scale(A, scalar):
    """Perkalian matriks dengan skalar."""
    return [[A[i][j] * scalar for j in range(len(A[0]))] for i in range(len(A))]



def mean_columns(data):
    """
    Hitung rata-rata setiap kolom (fitur).
    data: list of list, ukuran [n_samples x n_features]
    """
    n = len(data)
    n_features = len(data[0])
    return [sum(data[i][j] for i in range(n)) / n for j in range(n_features)]


def center_data(data):
    """
    Pusatkan data dengan mengurangi mean setiap kolom.
    Mengembalikan (data_centered, mean_vec)
    """
    means = mean_columns(data)
    centered = [[data[i][j] - means[j] for j in range(len(data[0]))] for i in range(len(data))]
    return centered, means


def covariance_matrix(data_centered):
    """
    Hitung covariance matrix dari data yang sudah dipusatkan.
    Cov = (1 / (n-1)) * X^T * X
    """
    n = len(data_centered)
    X_T = transpose(data_centered)
    cov = mat_mul(X_T, data_centered)

    return mat_scale(cov, 1.0 / (n - 1))



def power_iteration(cov, num_iterations=1000, tol=1e-10):
    """
    Hitung eigenvector dominan (terbesar) menggunakan Power Iteration.

    Ide: vektor acak dikalikan berulang dengan matriks cov,
         lama-kelamaan akan konvergen ke eigenvector dengan eigenvalue terbesar.
    """
    n = len(cov)

    b = [1.0 / (n ** 0.5)] * n

    for _ in range(num_iterations):
        b_new = mat_vec_mul(cov, b)
        b_new_norm = vec_norm(b_new)
        if b_new_norm < 1e-15:
            break
        b_new = vec_normalize(b_new)

        diff = vec_norm(vec_sub(b_new, b))
        b = b_new
        if diff < tol:
            break

    Av = mat_vec_mul(cov, b)
    eigenvalue = vec_dot(b, Av)
    return eigenvalue, b


def deflate(cov, eigenvalue, eigenvector):
    """
    Deflasi matriks: hapus komponen eigenvector yang sudah ditemukan.
    A_new = A - eigenvalue * (v * v^T)
    """
    rank1 = outer_product(eigenvector, eigenvector)
    rank1_scaled = mat_scale(rank1, eigenvalue)
    return mat_sub(cov, rank1_scaled)


def eigen_decomposition(cov, n_components):
    """
    Hitung n_components eigenvalue dan eigenvector terbesar
    menggunakan Power Iteration + Deflation secara berulang.
    """
    eigenvalues = []
    eigenvectors = []
    cov_deflated = [row[:] for row in cov]  # copy

    for _ in range(n_components):
        val, vec = power_iteration(cov_deflated)
        eigenvalues.append(val)
        eigenvectors.append(vec)
        cov_deflated = deflate(cov_deflated, val, vec)

    return eigenvalues, eigenvectors



class PCA:
    """
    Principal Component Analysis (PCA) dari Scratch.

    Parameter:
    ----------
    n_components : int
        Jumlah principal component yang ingin dipertahankan.

    Atribut setelah fit():
    ----------------------
    mean_           : list, mean setiap fitur
    components_     : list of list, eigenvector (PC axes), ukuran [n_components x n_features]
    explained_variance_      : list, eigenvalue tiap komponen
    explained_variance_ratio_: list, proporsi varians yang dijelaskan
    """

    def __init__(self, n_components):
        self.n_components = n_components
        self.mean_ = None
        self.components_ = None
        self.explained_variance_ = None
        self.explained_variance_ratio_ = None

    def fit(self, X):
        """
        Hitung PCA dari data X.
        X: list of list, ukuran [n_samples x n_features]
        """

        X_centered, self.mean_ = center_data(X)


        cov = covariance_matrix(X_centered)


        eigenvalues, eigenvectors = eigen_decomposition(cov, self.n_components)

        self.explained_variance_ = eigenvalues
        self.components_ = eigenvectors

        total_var = sum(eigenvalues)
        if total_var > 0:
            self.explained_variance_ratio_ = [v / total_var for v in eigenvalues]
        else:
            self.explained_variance_ratio_ = [0.0] * len(eigenvalues)

        return self

    def transform(self, X):
        """
        Proyeksikan data X ke ruang principal component.
        Mengembalikan list of list ukuran [n_samples x n_components]
        """
        assert self.components_ is not None, "Jalankan fit() terlebih dahulu."
        # Pusatkan X dengan mean dari training
        X_centered = [[X[i][j] - self.mean_[j] for j in range(len(X[0]))] for i in range(len(X))]
        # Proyeksi: X_centered @ components_^T
        result = []
        for sample in X_centered:
            projected = [vec_dot(sample, pc) for pc in self.components_]
            result.append(projected)
        return result

    def fit_transform(self, X):
        """Fit lalu langsung transform."""
        self.fit(X)
        return self.transform(X)

    def inverse_transform(self, X_transformed):
        """
        Rekonstruksi data dari ruang PC kembali ke ruang asli.
        """
        assert self.components_ is not None, "Jalankan fit() terlebih dahulu."
        reconstructed = []
        for sample in X_transformed:
            # Gabungkan: sum(score_k * pc_k) + mean
            rec = [self.mean_[j] for j in range(len(self.mean_))]
            for k, score in enumerate(sample):
                pc = self.components_[k]
                for j in range(len(rec)):
                    rec[j] += score * pc[j]
            reconstructed.append(rec)
        return reconstructed

    def summary(self):
        """Tampilkan ringkasan hasil PCA."""
        print("=" * 55)
        print("           HASIL PCA ")
        print("=" * 55)
        print(f"Jumlah komponen  : {self.n_components}")
        print(f"Mean fitur       : {[round(m, 4) for m in self.mean_]}")
        print()
        cumulative = 0.0
        for i, (val, ratio) in enumerate(
            zip(self.explained_variance_, self.explained_variance_ratio_)
        ):
            cumulative += ratio
            print(f"  PC{i+1}:")
            print(f"    Eigenvalue (Varians)    : {val:.6f}")
            print(f"    Explained Variance Ratio: {ratio*100:.2f}%")
            print(f"    Kumulatif               : {cumulative*100:.2f}%")
            print(f"    Eigenvector (loading)   : [{', '.join(f'{x:.4f}' for x in self.components_[i])}]")
            print()
        print("=" * 55)



def main():
    print("=" * 55)
    print("  DEMO PCA dari Scratch")
    print("=" * 55)


    data = [
        [5.1, 3.5, 1.4, 0.2],
        [4.9, 3.0, 1.4, 0.2],
        [6.2, 3.4, 5.4, 2.3],
        [5.9, 3.0, 5.1, 1.8],
        [6.7, 3.1, 4.7, 1.5],
        [5.5, 2.5, 4.0, 1.3],
        [4.6, 3.4, 1.4, 0.3],
        [6.3, 3.3, 6.0, 2.5],
    ]

    print(f"\nData asli ({len(data)} sampel, {len(data[0])} fitur):")
    for i, row in enumerate(data):
        print(f"  Sampel {i+1}: {row}")


    pca = PCA(n_components=2)
    X_transformed = pca.fit_transform(data)


    print()
    pca.summary()


    print("\nData setelah transformasi ke 2 PC:")
    for i, row in enumerate(X_transformed):
        print(f"  Sampel {i+1}: PC1={row[0]:.4f}, PC2={row[1]:.4f}")


    X_reconstructed = pca.inverse_transform(X_transformed)
    print("\nData rekonstruksi (approx) dari 2 PC:")
    for i, row in enumerate(X_reconstructed):
        print(f"  Sampel {i+1}: {[round(v, 4) for v in row]}")


    total_error = 0.0
    for orig, recon in zip(data, X_reconstructed):
        err = sum((o - r) ** 2 for o, r in zip(orig, recon)) ** 0.5
        total_error += err
    print(f"\nRata-rata Reconstruction Error (Euclidean): {total_error / len(data):.6f}")
    print("\nSelesai.")


if __name__ == "__main__":
    main()

  DEMO PCA dari Scratch

Data asli (8 sampel, 4 fitur):
  Sampel 1: [5.1, 3.5, 1.4, 0.2]
  Sampel 2: [4.9, 3.0, 1.4, 0.2]
  Sampel 3: [6.2, 3.4, 5.4, 2.3]
  Sampel 4: [5.9, 3.0, 5.1, 1.8]
  Sampel 5: [6.7, 3.1, 4.7, 1.5]
  Sampel 6: [5.5, 2.5, 4.0, 1.3]
  Sampel 7: [4.6, 3.4, 1.4, 0.3]
  Sampel 8: [6.3, 3.3, 6.0, 2.5]

           HASIL PCA 
Jumlah komponen  : 2
Mean fitur       : [5.65, 3.15, 3.675, 1.2625]

  PC1:
    Eigenvalue (Varians)    : 5.177965
    Explained Variance Ratio: 97.72%
    Kumulatif               : 97.72%
    Eigenvector (loading)   : [0.2974, -0.0198, 0.8640, 0.4058]

  PC2:
    Eigenvalue (Varians)    : 0.121056
    Explained Variance Ratio: 2.28%
    Kumulatif               : 100.00%
    Eigenvector (loading)   : [0.6749, 0.7108, -0.1902, -0.0549]


Data setelah transformasi ke 2 PC:
  Sampel 1: PC1=-2.5673, PC2=0.3687
  Sampel 2: PC1=-2.6168, PC2=-0.1217
  Sampel 3: PC1=2.0700, PC2=0.1638
  Sampel 4: PC1=1.5266, PC2=-0.2385
  Sampel 5: PC1=1.2952, PC2=0.4651
  